In [ ]:
!sudo apt-get install python3.10 python3.10-distutils -y
!wget https://bootstrap.pypa.io/get-pip.py
!python3.10 get-pip.py
!python3.10 -m pip install -q mediapipe-model-maker
!unzip -o -q Imagenes.zip -d /content/dataset

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Note, selecting 'python3-distutils' instead of 'python3.10-distutils'
python3-distutils is already the newest version (3.10.8-1~22.04).
python3.10 is already the newest version (3.10.12-1~22.04.14).
0 upgraded, 0 newly installed, 0 to remove and 51 not upgraded.
--2026-02-11 07:49:48--  https://bootstrap.pypa.io/get-pip.py
Resolving bootstrap.pypa.io (bootstrap.pypa.io)... 151.101.0.175, 151.101.64.175, 151.101.128.175, ...
Connecting to bootstrap.pypa.io (bootstrap.pypa.io)|151.101.0.175|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2193439 (2.1M) [text/x-python]
Saving to: ‘get-pip.py.3’

get-pip.py.3        100%[===================>]   2.09M  --.-KB/s    in 0.07s   

2026-02-11 07:49:48 (29.0 MB/s) - ‘get-pip.py.3’ saved [2193439/2193439]

  Using cached pip-26.0.1-py3-none-any.whl.metadata (4.7 kB)
Using cached pip-26.0.1-py3-none-any.whl (1.8 MB)
  Attemptin

In [ ]:
%%writefile entrenar.py
import os
from mediapipe_model_maker import image_classifier

DATASET_PATH = "/content/dataset/Imagenes"
data = image_classifier.Dataset.from_folder(DATASET_PATH)
train_data, validation_data = data.split(0.8)

print(f"Imágenes para entrenar: {len(train_data)}")
print(f"Imágenes para validar: {len(validation_data)}")

# --- LA CORRECCIÓN ESTÁ AQUÍ ---
hparams = image_classifier.HParams(epochs=15)
options = image_classifier.ImageClassifierOptions(
    supported_model=image_classifier.SupportedModels.EFFICIENTNET_LITE0,
    hparams=hparams
)
# -------------------------------

print("Iniciando entrenamiento...")
model = image_classifier.ImageClassifier.create(
    train_data=train_data,
    validation_data=validation_data,
    options=options,
)

loss, accuracy = model.evaluate(validation_data)
print(f"Precisión final: {accuracy * 100:.2f}%")

model.export_model("modelo_kirby.tflite")
print("¡Modelo exportado con éxito!")

Overwriting entrenar.py


In [ ]:
!MPLBACKEND=agg python3.10 entrenar.py

2026-02-11 07:56:59.430196: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-02-11 07:56:59.462948: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-02-11 07:56:59.463015: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-02-11 07:56:59.464233: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-02-11 07:56:59.469921: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-02-11 07:56:59.470164: I tensorflow/core/platform/cpu_feature_guard.cc:1

In [ ]:
import subprocess
from google.colab import files

print("Escaneando la totalidad del servidor (esto tomará unos segundos)...")

# Ejecutamos el comando 'find' de Linux desde la raíz absoluta '/'
resultado = subprocess.run(['find', '/', '-name', '*.tflite', '-type', 'f'], capture_output=True, text=True)

# Limpiamos los resultados
rutas = resultado.stdout.strip().split('\n')

# Filtramos para ignorar los modelos de ejemplo que vienen instalados dentro de la propia librería de MediaPipe
archivos_validos = [r for r in rutas if r and "site-packages" not in r and "dist-packages" not in r]

if archivos_validos:
    # Tomamos el archivo más reciente o el primero que encuentre
    ruta_exacta = archivos_validos[-1]
    print(f"¡BINGO! Tu modelo estaba escondido en: {ruta_exacta}")
    print("Iniciando la descarga a tu computadora...")

    # Forzar descarga
    files.download(ruta_exacta)
else:
    print("El escáner terminó. Definitivamente no hay ningún modelo generado en el servidor.")
    print("Si sale este mensaje, significa que el script 'entrenar.py' no logró ejecutar la última línea de exportación.")

Escaneando la totalidad del servidor (esto tomará unos segundos)...
¡BINGO! Tu modelo estaba escondido en: /tmp/tmpvc0z9xee/modelo_kirby.tflite
Iniciando la descarga a tu computadora...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
model.export_model("modelo_kirby.tflite")

NameError: name 'model' is not defined